In [ ]:
!pip install pyngrok h2o

In [ ]:
import h2o
import os
import pandas as pd
from pyngrok import ngrok
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

ngrok.kill()
ngrok.set_auth_token("token_ngrok")

try:
    h2o.init(ip="0.0.0.0", port=porta)
    print("H2O inicializado com sucesso.")
except Exception as e:
    print(f"Erro ao iniciar H2O: {e}")

public_url = ngrok.connect(porta)
print(f"\n ACESSE O FLOW AQUI: {public_url}")
print("Nota: Ao abrir, clique em 'Visit Site' na página de aviso do ngrok.\n")

df0 = pd.read_csv('data_processed.csv')
target = 'Risco_Doenca'

X = df0.drop(columns=[target])
y = df0[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

iso = IsolationForest(contamination=0.05, random_state=42)
outliers = iso.fit_predict(X_train.drop(columns=['ID'], errors='ignore'))

X_train_cleaned = X_train[outliers == 1].copy()
y_train_cleaned = y_train[outliers == 1].copy()

X_train_cleaned[target] = y_train_cleaned
print(f"Dados limpos: {X_train_cleaned.shape[0]} linhas restantes.")

import h2o
h2o.init()
h2o_df = h2o.H2OFrame(X_train_cleaned)
h2o_df[target] = h2o_df[target].asfactor()
print("\nTipos das colunas no H2O (Procure por 'enum' no target):")
print(h2o_df.types)
from h2o.automl import H2OAutoML
y = target
x = [col for col in h2o_df.columns if col not in [y, 'ID']]
print("Iniciando o AutoML... Isso vai levar alguns minutos.")
# Configurar o AutoML
aml = H2OAutoML(
    max_runtime_secs=600,
    seed=1,
    balance_classes=True,
    sort_metric="logloss", # Força métrica de classificação
    project_name="projetosaude"
)
aml.train(x=x, y=y, training_frame=h2o_df)
print("Treinamento concluído! Acesso ao H2O Flow para visualizar os resultados.")